# Delta Lake com PySpark
Demonstração de operações CRUD em tabelas Delta Lake usando dados de E-commerce.

In [1]:
import pyspark
from delta import *
from commons import get_raw_data, filter_southern_region

builder = pyspark.sql.SparkSession.builder.appName("DeltaLab") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print('Spark iniciado com sucesso!')

26/04/28 17:09:05 WARN Utils: Your hostname, NOTE-LAURA resolves to a loopback address: 127.0.1.1; using 172.17.40.94 instead (on interface eth0)
26/04/28 17:09:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/lauras/.cache/pypoetry/virtualenvs/spark-delta-iceberg-Qz15G4zY-py3.12/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/lauras/.ivy2/cache
The jars for the packages stored in: /home/lauras/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-540d8fcf-81e9-4c9d-b543-4d2f9553a93f;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.0/delta-spark_2.12-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.0!delta-spark_2.12.jar (514ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.2.0/delta-storage-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.2.0!delta-storage.jar (72ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (89ms)
:: resolution report :: resolve 3157ms :: a

Spark iniciado com sucesso!


## Carregando os dados

In [2]:
df_vendas = get_raw_data(spark, "vendas.csv")
df_clientes = get_raw_data(spark, "clientes.csv")
df_vendas.show(5)
df_clientes.show(5)

+--------+----------+----------+------+---------+
|id_venda|id_cliente|data_venda| valor|   status|
+--------+----------+----------+------+---------+
|       1|       101|2023-01-15| 250.5|concluido|
|       2|       102|2023-01-16| 120.0|concluido|
|       3|       103|2023-01-17| 340.9|cancelado|
|       4|       101|2023-01-18|  50.0|concluido|
|       5|       104|2023-01-19|1500.0|concluido|
+--------+----------+----------+------+---------+

+----------+--------------+------+
|id_cliente|          nome|estado|
+----------+--------------+------+
|       101|     Ana Julia|    SC|
|       102|Laura Silveira|    RS|
|       103|  Carlos Silva|    SP|
|       104|  Mateus Souza|    PR|
+----------+--------------+------+



## INSERT — Salvando dados no formato Delta Lake

In [3]:
df_sul = filter_southern_region(df_vendas.join(df_clientes, "id_cliente"))
df_sul.write.format("delta").mode("overwrite").save("../data/delta/vendas_sul")
print('Dados salvos no Delta Lake!')
spark.read.format("delta").load("../data/delta/vendas_sul").show()

Dados salvos no Delta Lake!


26/04/28 17:09:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+--------+----------+------+---------+--------------+------+
|id_cliente|id_venda|data_venda| valor|   status|          nome|estado|
+----------+--------+----------+------+---------+--------------+------+
|       101|       1|2023-01-15| 250.5|concluido|     Ana Julia|    SC|
|       102|       2|2023-01-16| 120.0|concluido|Laura Silveira|    RS|
|       101|       4|2023-01-18|  50.0|concluido|     Ana Julia|    SC|
|       104|       5|2023-01-19|1500.0|concluido|  Mateus Souza|    PR|
+----------+--------+----------+------+---------+--------------+------+



## UPDATE — Atualizando valor da venda

In [4]:
spark.read.format("delta").load("../data/delta/vendas_sul").createOrReplaceTempView("vendas_delta")
spark.sql("UPDATE vendas_delta SET valor = valor * 1.10 WHERE id_venda = 1")
print('UPDATE realizado!')
spark.sql("SELECT * FROM vendas_delta").show()

UPDATE realizado!
+----------+--------+----------+------+---------+--------------+------+
|id_cliente|id_venda|data_venda| valor|   status|          nome|estado|
+----------+--------+----------+------+---------+--------------+------+
|       101|       1|2023-01-15|275.55|concluido|     Ana Julia|    SC|
|       102|       2|2023-01-16| 120.0|concluido|Laura Silveira|    RS|
|       101|       4|2023-01-18|  50.0|concluido|     Ana Julia|    SC|
|       104|       5|2023-01-19|1500.0|concluido|  Mateus Souza|    PR|
+----------+--------+----------+------+---------+--------------+------+



## DELETE — Removendo uma venda

In [5]:
spark.sql("DELETE FROM vendas_delta WHERE id_venda = 3")
print('DELETE realizado!')
spark.sql("SELECT * FROM vendas_delta").show()

DELETE realizado!
+----------+--------+----------+------+---------+--------------+------+
|id_cliente|id_venda|data_venda| valor|   status|          nome|estado|
+----------+--------+----------+------+---------+--------------+------+
|       101|       1|2023-01-15|275.55|concluido|     Ana Julia|    SC|
|       102|       2|2023-01-16| 120.0|concluido|Laura Silveira|    RS|
|       101|       4|2023-01-18|  50.0|concluido|     Ana Julia|    SC|
|       104|       5|2023-01-19|1500.0|concluido|  Mateus Souza|    PR|
+----------+--------+----------+------+---------+--------------+------+

